# vi-function-calling-slm · 03 DPO trên model SFT

Cần **GPU T4**. Kaggle: Settings → Accelerator → GPU T4 x2. Colab: Runtime → Change runtime type → T4 GPU.
Input: code + output notebook 01 (dữ liệu) + output notebook 02 (`vi-fc-sft-lora`); trên Colab tất cả nằm
sẵn trên Drive. Secret: `HF_TOKEN` (tuỳ chọn). Colab bị ngắt thì chạy lại notebook: bản merge SFT, lỗi on-policy và
checkpoint DPO (`outputs/dpo-<tag adapter SFT>`) đều lưu trên Drive nên không phải làm lại từ đầu.

**Bẫy model tham chiếu:** DPOTrainer với PEFT và `ref_model=None` lấy model *tắt adapter* làm tham chiếu.
Nếu cứ load adapter SFT rồi train tiếp thì tắt adapter = Qwen3 gốc, không phải model SFT -> DPO kéo về sai mốc.
Nên: gộp adapter SFT vào trọng số (merge 16-bit) -> load lại 4-bit -> gắn LoRA mới cho DPO.
Khi đó tắt adapter = đúng model SFT.

Cặp preference: lỗi thật của model SFT trên 300 câu train (on-policy) + nhiễu có chủ đích (notebook 01),
số cặp chốt theo ngân sách ~20 phút train trên T4 (quá 25 phút thì dừng cứng). beta = 0.1, lr 1e-5 (ít step nên lr cao hơn mức hay dùng 5e-6), 1 epoch.
Có Unsloth thì dùng, không thì transformers + peft + bitsandbytes.

In [ ]:
import gc, glob, json, os, random, sys

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")  # Kaggle T4 x2: train trên 1 GPU
IN_COLAB = "google.colab" in sys.modules
# Colab bản mới cũng có sẵn /kaggle/input (tích hợp dataset Kaggle) -> nhận Kaggle bằng biến môi trường riêng của nó
IN_KAGGLE = not IN_COLAB and "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if IN_KAGGLE:
    CODE = glob.glob("/kaggle/input/**/vi_fc/__init__.py", recursive=True)[0].rsplit("/vi_fc", 1)[0]
    REPO = CODE.rsplit("/src", 1)[0]
    WORK = "/kaggle/working"
else:
    # Colab / máy local: lấy code từ GitHub, kết quả để trên Drive cho khỏi mất khi runtime bị ngắt
    REPO = "/content/vi-function-calling-slm" if IN_COLAB else os.path.abspath("vi-function-calling-slm")
    if not os.path.exists(REPO):
        !git clone --depth 1 https://github.com/DuongCodeAI/vi-function-calling-slm {REPO}
    else:  # đã clone ở lần chạy trước -> lấy bản code mới nhất
        !git -C {REPO} pull -q
    CODE = f"{REPO}/src"
    WORK = os.getcwd()
    if IN_COLAB:
        from google.colab import drive
        try:  # không ai bấm cho phép Drive thì sau 2 phút chạy trên /content, output trước lấy từ HF Hub
            drive.mount("/content/drive", timeout_ms=120_000)
            WORK = "/content/drive/MyDrive/ai-portfolio/vi-function-calling-slm"
        except Exception as e:
            print("không mount được Drive, chạy trên /content/work:", e)
            WORK = "/content/work"
NO_DRIVE = WORK == "/content/work"
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, CODE)


def secret(name):
    # Kaggle Secrets / Colab Secrets / biến môi trường, thiếu thì trả None
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)


def find(pat, hint):
    # output của notebook trước: Kaggle thì add làm input (/kaggle/input), Colab thì nằm sẵn trong WORK trên Drive
    for root in (["/kaggle/input"] if IN_KAGGLE else []) + [WORK, REPO]:
        hits = glob.glob(f"{root}/{pat}") or glob.glob(f"{root}/**/{pat}", recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"không tìm thấy {pat}: {hint}")


if NO_DRIVE and not os.path.exists(f"{WORK}/data/gen/train.jsonl"):
    from vi_fc.hub import pull
    os.environ.setdefault("HF_TOKEN", secret("HF_TOKEN") or "")
    pull("vi-fc-synth", f"{WORK}/data/gen", repo_type="dataset")  # notebook 01 đã đẩy lên (repo private)
    pull("vi-fc-qwen3-1.7b", f"{WORK}/vi-fc-sft-lora", subdir="sft-lora")  # adapter notebook 02 đã đẩy

DATA = os.path.dirname(find("data/gen/train.jsonl", "chạy notebook 01 trước"))
SFT_DIR = os.path.dirname(find("vi-fc-sft-lora/adapter_config.json", "chạy notebook 02 trước"))
print(CODE, DATA, SFT_DIR)
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
# khoá bản torch có sẵn: pip mà tự nâng torch thì torchvision/torchaudio lệch bản, import transformers hỏng theo
!pip list --format=freeze | grep -iE "^(torch|torchvision|torchaudio)==" > /tmp/keep-torch.txt
!timeout 900 pip install -q -c /tmp/keep-torch.txt unsloth || echo "cài unsloth lỗi, dùng transformers + peft"
!pip install -q -c /tmp/keep-torch.txt bitsandbytes peft trl datasets accelerate

token = secret("HF_TOKEN")
if token:
    os.environ["HF_TOKEN"] = token
else:
    print("thiếu HF_TOKEN, sẽ không push")
if secret("HF_USER"):  # tuỳ chọn: đẩy vào tài khoản/org khác với chủ HF_TOKEN
    os.environ["HF_USER"] = secret("HF_USER")

from vi_fc.train import has_unsloth

UNSLOTH = has_unsloth()
print("unsloth:", UNSLOTH)

In [ ]:
if UNSLOTH:
    import unsloth

    if hasattr(unsloth, "PatchDPOTrainer"):  # bản mới tự patch TRL lúc import, hàm này thành thừa
        unsloth.PatchDPOTrainer()
import hashlib, shutil

import torch
from vi_fc.train import load_4bit, merge_lora

MAX_LEN = 2048  # prompt ~1.7k token (system + 17 tool), câu trả lời < 100
MERGED = f"{WORK}/sft-merged"
# 1) gộp adapter SFT vào Qwen3 fp16. Bản merge gắn với đúng adapter: notebook 02 train lại thì merge lại,
#    không dùng nhầm bản cũ còn trên Drive
TAG = hashlib.md5(open(glob.glob(f"{SFT_DIR}/adapter_model.*")[0], "rb").read()).hexdigest()[:8]
if not os.path.exists(f"{MERGED}/sft_tag.txt") or open(f"{MERGED}/sft_tag.txt").read() != TAG:
    shutil.rmtree(MERGED, ignore_errors=True)
    merge_lora(SFT_DIR, MERGED)
    open(f"{MERGED}/sft_tag.txt", "w").write(TAG)
RUN = f"outputs/dpo-{TAG}"
os.makedirs(RUN, exist_ok=True)

# 2) load lại bản merge ở 4-bit
model, tokenizer = load_4bit(MERGED, MAX_LEN, UNSLOTH)

## Lỗi on-policy: chạy model SFT trên 300 câu train, giữ câu sai làm rejected

In [ ]:
from vi_fc.dpo import mine_on_policy
from vi_fc.evaluate import compute_metrics, load_jsonl, predict
from vi_fc.inference import FunctionCaller
from vi_fc.train import set_mode

train = load_jsonl(f"{DATA}/train.jsonl")
sub = random.Random(0).sample(train, min(300, len(train)))
# lưu ra file: Colab bị ngắt thì lần sau đọc lại, cặp DPO giữ nguyên để resume checkpoint cho đúng
mined = f"{RUN}/sft_train{len(sub)}.jsonl"
rows = load_jsonl(mined) if os.path.exists(mined) else []
if len(rows) < len(sub):
    set_mode(model, UNSLOTH, train=False)
    fc = FunctionCaller("hf", model, tokenizer=tokenizer, guard=False, max_tokens=128)
    rows = predict(sub, fc, "sft", "train", mined)
print(f"SFT trên {len(sub)} câu train:",
      {k: v for k, v in compute_metrics(rows).items() if k != "exact_by_scenario"})
on_policy = mine_on_policy(sub, rows)  # rows có cả raw: output hỏng format giữ nguyên làm rejected
print(len(on_policy), "cặp on-policy")

In [ ]:
import statistics
from collections import Counter

from datasets import Dataset
from vi_fc.prompt import split_prompt_completion
from vi_fc.tools import openai_tools
from vi_fc.train import TPS, plan_steps

TOOLS = openai_tools()


def apply(msgs, gen):
    return tokenizer.apply_chat_template(msgs, tools=TOOLS, tokenize=False, add_generation_prompt=gen,
                                         enable_thinking=False)


def render(p):
    prompt, chosen = split_prompt_completion(apply, p["messages"] + [p["chosen"]])
    _, rejected = split_prompt_completion(apply, p["messages"] + [p["rejected"]])
    # TRL tự nối eos (<|im_end|>) vào chosen/rejected -> bỏ cái template đã thêm cho khỏi lặp
    end = tokenizer.eos_token + "\n"
    return {"prompt": prompt, "chosen": chosen.removesuffix(end), "rejected": rejected.removesuffix(end)}


programmatic = load_jsonl(f"{DATA}/dpo_train.jsonl")
random.Random(0).shuffle(programmatic)
BS, GA, DPO_MIN = 1, 8, 20
# số cặp theo ngân sách 20 phút: mỗi cặp = chosen + rejected, mỗi chuỗi policy forward + backward và ref forward
# ≈ 2.5 lần một ví dụ SFT. Ưu tiên lỗi on-policy, phần còn lại lấy nhiễu có chủ đích
seq = statistics.mean(len(tokenizer(r["prompt"] + r["chosen"]).input_ids)
                      for r in map(render, (on_policy + programmatic)[:20]))
cap = plan_steps(10**6, BS * GA, 1, 2.5 * seq, TPS[UNSLOTH], DPO_MIN) * BS * GA
pairs = on_policy[:cap] + programmatic[:max(0, cap - len(on_policy))]
print(len(pairs), Counter(p["type"] for p in pairs).most_common())

ds = Dataset.from_list([render(p) for p in pairs]).shuffle(seed=0)
print(ds[0]["chosen"]); print("---"); print(ds[0]["rejected"])

In [ ]:
import dataclasses, math

from trl import DPOConfig, DPOTrainer
from vi_fc.train import add_lora, fp_flags, time_budget

model = add_lora(model, UNSLOTH)
steps = math.ceil(len(ds) / (BS * GA))
kw = dict(
    output_dir=RUN, beta=0.1, learning_rate=1e-5, num_train_epochs=1, lr_scheduler_type="cosine",
    warmup_steps=max(1, steps // 20), per_device_train_batch_size=BS, gradient_accumulation_steps=GA,
    max_length=MAX_LEN, max_prompt_length=MAX_LEN, max_completion_length=256,
    optim="adamw_8bit", **fp_flags(), logging_steps=5, report_to="none", seed=0,
    save_strategy="steps" if IN_COLAB else "no", save_steps=20, save_total_limit=1,  # Colab: checkpoint lên Drive
)
# TRL hay đổi tên / bỏ tham số giữa các bản -> chỉ truyền những gì DPOConfig bản đang cài có
fields = {f.name for f in dataclasses.fields(DPOConfig)}
print("DPOConfig không có, bỏ qua:", sorted(set(kw) - fields))
args = DPOConfig(**{k: v for k, v in kw.items() if k in fields})
trainer = DPOTrainer(model=model, ref_model=None, args=args, train_dataset=ds, processing_class=tokenizer,
                     callbacks=[time_budget(DPO_MIN + 5)])
trainer.train(resume_from_checkpoint=bool(glob.glob(f"{RUN}/checkpoint-*")))
model.save_pretrained("vi-fc-dpo-lora")
tokenizer.save_pretrained("vi-fc-dpo-lora")

In [ ]:
import matplotlib.pyplot as plt

hist = [h for h in trainer.state.log_history if "rewards/margins" in h]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot([h["step"] for h in hist], [h["rewards/margins"] for h in hist]); ax[0].set_title("reward margin")
ax[1].plot([h["step"] for h in hist], [h.get("rewards/accuracies") for h in hist]); ax[1].set_title("reward accuracy")
fig.savefig("dpo_rewards.png", dpi=110, bbox_inches="tight")

## Gộp adapter DPO vào bản SFT (fp16) cho notebook 04

In [ ]:
trainer = model = fc = None  # nhả VRAM (fc lúc đào lỗi on-policy cũng giữ model)
gc.collect(); torch.cuda.empty_cache()
merge_lora("vi-fc-dpo-lora", f"{WORK}/dpo-merged", base=MERGED)

from vi_fc.hub import push

push("vi-fc-dpo-lora", "vi-fc-qwen3-1.7b", path_in_repo="dpo-lora")  # lỗi hay thiếu token thì chỉ in ra
!du -sh sft-merged dpo-merged vi-fc-dpo-lora

## SFT+DPO trên val + bộ viết tay (so với số SFT ở notebook 02)

In [ ]:
fc = FunctionCaller("hf", f"{WORK}/dpo-merged", guard=False, max_tokens=128)
val = load_jsonl(f"{DATA}/val.jsonl")[:50]
manual = load_jsonl(find("data/test_manual.jsonl", "thiếu data/test_manual.jsonl của repo"))
for name, recs in (("val", val), ("manual", manual)):
    m = compute_metrics(predict(recs, fc, "sft-dpo", name, f"dpo_{name}.jsonl"))
    print(name, json.dumps({k: v for k, v in m.items() if k != "exact_by_scenario"}, ensure_ascii=False))